In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, recall_score, confusion_matrix, roc_auc_score, make_scorer, roc_curve
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV, ParameterGrid
from sklearn.calibration import calibration_curve, CalibratedClassifierCV
import xgboost as xgb
from sklearn.utils import resample

In [ ]:
from sdv.single_table import TVAESynthesizer
from sdv.metadata import Metadata

In [ ]:
fig_save_dir = r"D:\Singapore\Alzheimer\R files\final data_new\figures\TVAE"
os.makedirs(fig_save_dir, exist_ok=True)

In [ ]:
# Load the Excel file
file_path =  r"D:\Singapore\Alzheimer\R files\final data_new\Variable list-11MAY2025.xlsx"
df_Variables = pd.read_excel(file_path)

In [ ]:
save_dir = r"D:\Singapore\Alzheimer\R files\final data_new\py"

In [ ]:
file_path_1 = r"D:\Singapore\Alzheimer\R files\final data_new\imoutations_diff_ks\final_data_imputed_k1.csv"
df_Data = pd.read_csv(file_path_1)
df_Data.info()

In [ ]:
# Group features based on the "Group" column
demographics = df_Variables[df_Variables['Group'] == 'Demographic']['Predictor'].tolist()
clinical_cognitive = df_Variables[df_Variables['Group'] == 'Clinical/Cognitive']['Predictor'].tolist()
biomarker = df_Variables[df_Variables['Group'] == 'Biomarker']['Predictor'].tolist()
neuroimaging = df_Variables[df_Variables['Group'] == 'Neuroimaging']['Predictor'].tolist()

# Optional: Print or check lists
print("Demographics:", demographics)
print("Clinical/Cognitive:", clinical_cognitive)
print("Biomarker:", biomarker)
print("Neuroimaging:", neuroimaging)

In [ ]:
# Group combinations
group_1 = demographics + clinical_cognitive
group_2 = demographics + biomarker
group_3 = demographics + clinical_cognitive + biomarker
group_4 = neuroimaging

In [ ]:
print(f'group_1:\n', group_1)
print('*'*100)
print(f'group_2:\n', group_2)
print('*'*100)
print(f'group_3:\n', group_3)
print('*'*100)
print(f'group_4:\n', group_4)

In [ ]:
# TVAE + RF and TVAE + XGB for group_1 and group_3

def compute_confidence_interval(metric_fn, y_true, y_score, threshold=None, n_bootstraps=1000, ci=0.95):
    stats = []
    y_true = np.array(y_true)
    y_score = np.array(y_score)
    rng = np.random.RandomState(100)

    for _ in range(n_bootstraps):
        indices = rng.randint(0, len(y_true), len(y_true))
        if len(np.unique(y_true[indices])) < 2:
            continue

        if threshold is not None:
            y_pred = (y_score[indices] >= threshold).astype(int)
            value = metric_fn(y_true[indices], y_pred)
        else:
            value = metric_fn(y_true[indices], y_score[indices])

        stats.append(value)

    lower = np.percentile(stats, (1 - ci) / 2 * 100)
    upper = np.percentile(stats, (1 + ci) / 2 * 100)
    return lower, upper


def generate_synthetic_and_run_model(group_columns, df, group_name,
                                     target_col="convert_Within_3Years",
                                     method="RF",  # "RF" or "XGBOOST"
                                     test_size=0.3,
                                     random_state=100,
                                     sample_multiplier=[1, 2, 3]):
    if group_name not in ["group_1", "group_3"]:
        raise ValueError(f"TVAE augmentation is only supported for 'group_1' and 'group_3'. Got: {group_name}")

    real_data_group = pd.concat([df[group_columns], df[target_col]], axis=1)
    train_data, test_data = train_test_split(
        real_data_group,
        test_size=test_size,
        random_state=random_state,
        stratify=real_data_group[target_col]
    )

    np.random.seed(42)
    torch.manual_seed(42)

    metadata = Metadata.detect_from_dataframe(train_data)
    

    print(f"Using TVAE for {group_name}")
    tvae_params = {
        'embedding_dim': 64,
        'compress_dims': (128, 64),
        'decompress_dims': (64, 128),
        'l2scale': 1e-5,
        'batch_size': 64,
        'epochs': 100,
        'loss_factor': 0.5,
        'enforce_min_max_values': True,
        'enforce_rounding': True,
        'enable_gpu': True
    }

    synthesizer = TVAESynthesizer(metadata=metadata, **tvae_params)
    synthesizer.fit(train_data)
    torch_model = synthesizer._model
# Get real training data (features only)
    X_train_real = train_data[group_columns]
    X_numpy = X_train_real.values.astype(np.float32)
    X_tensor = torch.tensor(X_numpy)

# Use transformer.transform to get latent vectors
    with torch.no_grad():
        latent_vectors = torch_model.transformer.transform(X_tensor)

    latent_vectors = latent_vectors.numpy()
    print(f"Latent vectors shape: {latent_vectors.shape}")  # Expected: (n_samples, 64)

# Save for UMAP visualization
    np.save(os.path.join(save_dir, f"latent_vectors_{group_name}.npy"), latent_vectors)




    results = []

    for mult in sample_multiplier:
        synthetic_data = synthesizer.sample(num_rows=len(train_data) * mult)
        save_dir = r"D:\Singapore\Alzheimer\R files\final data_new\synthetic_data\TVAE\group_1_3"        
        filename = f"synthetic_{group_name}_TVAE_{mult}x.csv"
        save_path = os.path.join(save_dir, filename)
        synthetic_data.to_csv(save_path, index=False)

        augmented_train = pd.concat([train_data, synthetic_data], ignore_index=True)

        X_train_aug = augmented_train[group_columns]
        y_train_aug = augmented_train[target_col]
        X_test = test_data[group_columns]
        y_test = test_data[target_col]

        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=100)

        if method.upper() == "RF":
            model = RandomForestClassifier(random_state=100, class_weight='balanced')
            param_dist = {
                'n_estimators': [150, 200],
                'max_depth': [2, 3],
                'min_samples_split': [10, 15, 20],
                'min_samples_leaf': [6, 8, 10],
                'max_features': ['log2'],
                'bootstrap': [True],
                'max_samples': [0.5, 0.7, 0.9]
            }
        elif method.upper() == "XGBOOST":
            neg_count = (y_train_aug == 0).sum()
            pos_count = (y_train_aug == 1).sum()
            scale_pos_weight = neg_count / pos_count
            model = xgb.XGBClassifier(eval_metric='logloss', random_state=100, scale_pos_weight=scale_pos_weight)
            param_dist = {
            'n_estimators': [100, 200, 300],
            'max_depth': [2, 3, 4, 5],
            'learning_rate': [0.01, 0.05, 0.1],
            'min_child_weight': [1, 3, 5],
            'subsample': [0.6, 0.8, 1],
            'colsample_bytree': [0.6, 0.8, 1],
            'gamma': [0, 0.1, 0.3],
            'reg_alpha': [0, 0.5, 1],
            'reg_lambda': [1, 2, 5]}
                
                
        else:
            raise ValueError(f"Invalid method '{method}'. Choose 'RF' or 'XGBoost'.")

        def specificity_score(y_true, y_pred):
            tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
            return tn / (tn + fp) if (tn + fp) > 0 else 0

        scoring = {
            'roc_auc': 'roc_auc',
            'f1': make_scorer(f1_score),
            'recall': make_scorer(recall_score),
            'specificity': make_scorer(specificity_score)
        }

        search = RandomizedSearchCV(
            model,
            param_distributions=param_dist,
            cv=cv,
            scoring=scoring,
            refit='roc_auc',
            random_state=100
        )

        search.fit(X_train_aug, y_train_aug)
        best_params = search.best_params_

        uncalibrated_model = xgb.XGBClassifier(**best_params, random_state=100,  scale_pos_weight=scale_pos_weight) if method.upper() == "XGBOOST" else RandomForestClassifier(**best_params, random_state=100, class_weight='balanced')
        calibrated_model = CalibratedClassifierCV(uncalibrated_model, method='isotonic', cv=3)
        calibrated_model.fit(X_train_aug, y_train_aug)

        y_train_proba = calibrated_model.predict_proba(X_train_aug)[:, 1]
        y_test_proba = calibrated_model.predict_proba(X_test)[:, 1]

        youden_thresholds = []
        for train_idx, val_idx in cv.split(X_train_aug, y_train_aug):
            X_tr, X_val = X_train_aug.iloc[train_idx], X_train_aug.iloc[val_idx]
            y_tr, y_val = y_train_aug.iloc[train_idx], y_train_aug.iloc[val_idx]

            if method.upper() == "XGBOOST":
                model_cv = xgb.XGBClassifier(**best_params, random_state=100,  scale_pos_weight=scale_pos_weight)
            else:
                model_cv = RandomForestClassifier(**best_params, random_state=100, class_weight='balanced')

            
            model_cv_calibrated = CalibratedClassifierCV(model_cv, method='isotonic', cv=3)
            model_cv_calibrated.fit(X_tr, y_tr)
            y_val_proba = model_cv_calibrated.predict_proba(X_val)[:, 1]
            fpr, tpr, thresholds = roc_curve(y_val, y_val_proba)
            youden = tpr - fpr
            best_thresh = thresholds[np.argmax(youden)]
            youden_thresholds.append(best_thresh)

        optimal_threshold = np.mean(youden_thresholds)
        

        y_train_pred = (y_train_proba >= optimal_threshold).astype(int)
        y_test_pred = (y_test_proba >= optimal_threshold).astype(int)

        test_f1 = f1_score(y_test, y_test_pred)
        test_recall = recall_score(y_test, y_test_pred)
        tn_test, fp_test, fn_test, tp_test = confusion_matrix(y_test, y_test_pred).ravel()
        test_specificity = tn_test / (tn_test + fp_test)
        test_auc = roc_auc_score(y_test, y_test_proba)

        train_f1 = f1_score(y_train_aug, y_train_pred)
        train_recall = recall_score(y_train_aug, y_train_pred)
        tn_train, fp_train, fn_train, tp_train = confusion_matrix(y_train_aug, y_train_pred).ravel()
        train_specificity = tn_train / (tn_train + fp_train)
        train_auc = roc_auc_score(y_train_aug, y_train_proba)

        cv_results = search.cv_results_
        best_idx = search.best_index_
        cv_auc = cv_results['mean_test_roc_auc'][best_idx]
        cv_f1 = cv_results['mean_test_f1'][best_idx]
        cv_recall = cv_results['mean_test_recall'][best_idx]
        cv_specificity = cv_results['mean_test_specificity'][best_idx]

        ci_auc = compute_confidence_interval(roc_auc_score, y_test, y_test_proba, threshold=None)
        ci_f1 = compute_confidence_interval(f1_score, y_test, y_test_proba, threshold=optimal_threshold)
        ci_recall = compute_confidence_interval(recall_score, y_test, y_test_proba, threshold=optimal_threshold)
        ci_spec = compute_confidence_interval(specificity_score, y_test, y_test_proba, threshold=optimal_threshold)

        n_bins = 10
        n_bootstrap = 1000
        ci = 99
        bin_edges = np.linspace(0, 1, n_bins + 1)
        bin_centers = (bin_edges[1:] + bin_edges[:-1]) / 2
        bootstrap_curves = []
        rng = np.random.RandomState(100)
        for _ in range(n_bootstrap):
            indices = rng.randint(0, len(y_test), len(y_test))
            y_bs = y_test.iloc[indices]
            p_bs = y_test_proba[indices]
            
            if len(np.unique(y_bs)) < 2:
                continue
            try:
                pt_bs, pp_bs = calibration_curve(y_bs, p_bs, n_bins=n_bins, strategy='uniform')
                interp = np.interp(bin_centers, pp_bs, pt_bs)
                bootstrap_curves.append(interp)
            except:
                continue

        bootstrap_array = np.array(bootstrap_curves)
        lower_bound = np.percentile(bootstrap_array, (100 - ci) / 2, axis=0)
        upper_bound = np.percentile(bootstrap_array, 100 - (100 - ci) / 2, axis=0)

        prob_true, prob_pred = calibration_curve(y_test, y_test_proba, n_bins=n_bins, strategy='uniform')
        plt.figure(figsize=(6, 6))
        plt.plot(prob_pred, prob_true, marker='o', label='Calibrated Model')
        plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Perfect Calibration')
        plt.fill_between(bin_centers, lower_bound, upper_bound, color='blue', alpha=0.2, label='99% CI')
        plt.xlabel('Predicted Probability')
        plt.ylabel('True Probability')
        plt.title(f'Calibration Plot (Test Set) for {group_name}, {method}, Multiplier {mult}')
        plt.legend()
        plt.grid()
        plt.tight_layout()
        plt.savefig(os.path.join(fig_save_dir, f"calibration_{group_name}_{method}_Multiplier_{mult}.pdf"), dpi=600,bbox_inches="tight")
        plt.show()
        plt.close()

        results.append({
            'group': group_name,
            'method': method,
            'sample_multiplier': mult,
            'train_auc': train_auc,
            'cv_auc': cv_auc,
            'test_auc': test_auc,
            'train_f1': train_f1,
            'cv_f1': cv_f1,
            'test_f1': test_f1,
            'train_recall': train_recall,
            'cv_recall': cv_recall,
            'test_recall': test_recall,
            'train_specificity': train_specificity,
            'cv_specificity': cv_specificity,
            'test_specificity': test_specificity,
            'ci_auc': ci_auc,
            'ci_f1': ci_f1,
            'ci_recall': ci_recall,
            'ci_specificity': ci_spec,
            'best_params': search.best_params_
        })

    return results


In [ ]:
results_rf_group_1 = generate_synthetic_and_run_model(
    group_columns=group_1,
    df=df_Data,
    group_name="group_1",
    method="RF"
)

results_xgb_group_1 = generate_synthetic_and_run_model(
    group_columns=group_1,
    df=df_Data,
    group_name="group_1",
    method="XGBoost"
)

results_rf_group_3 = generate_synthetic_and_run_model(
    group_columns=group_3,
    df=df_Data,
    group_name="group_3",
    method="RF"
)

results_xgb_group_3 = generate_synthetic_and_run_model(
    group_columns=group_3,
    df=df_Data,
    group_name="group_3",
    method="XGBoost"
)



In [ ]:

all_results = [results_rf_group_1, results_xgb_group_1, results_rf_group_3, results_xgb_group_3]

In [ ]:
# Flatten results
summary_data = []
for res_list in all_results:
    for res in res_list:
        summary_data.append({
            "Group": res["group"],
            "Method": res["method"],
            "Multiplier": res["sample_multiplier"],

            "Train AUC": round(res["train_auc"], 3),
            "CV AUC": round(res["cv_auc"], 3),
            "Test AUC": round(res["test_auc"], 3),
            "AUC CI Lower": round(res["ci_auc"][0], 3),
            "AUC CI Upper": round(res["ci_auc"][1], 3),
        })

# Convert to DataFrame and display
summary_df_group_1_3_auc = pd.DataFrame(summary_data)
print(summary_df_group_1_3_auc.to_markdown(index=False))


In [ ]:
# Flatten results
summary_data = []
for res_list in all_results:
    for res in res_list:
        summary_data.append({
            "Group": res["group"],
            "Method": res["method"],
            "Multiplier": res["sample_multiplier"],

            "Train Recall": round(res["train_recall"], 3),
            "CV Recall": round(res["cv_recall"], 3),
            "Test Recall": round(res["test_recall"], 3),
            "Recall CI Lower": round(res["ci_recall"][0], 3),
            "Recall CI Upper": round(res["ci_recall"][1], 3),
        })

# Convert to DataFrame and display
summary_df_group_1_3_recall = pd.DataFrame(summary_data)
print(summary_df_group_1_3_recall.to_markdown(index=False))

In [ ]:
# Flatten results
summary_data = []
for res_list in all_results:
    for res in res_list:
        summary_data.append({
            "Group": res["group"],
            "Method": res["method"],
            "Multiplier": res["sample_multiplier"],

            "Train Specificity": round(res["train_specificity"], 3),
            "CV Specificity": round(res["cv_specificity"], 3),
            "Test Specificity": round(res["test_specificity"], 3),
            "Specificity CI Lower": round(res["ci_specificity"][0], 3),
            "Specificity CI Upper": round(res["ci_specificity"][1], 3),
        })

# Convert to DataFrame and display
summary_df_group_1_3_spec = pd.DataFrame(summary_data)
print(summary_df_group_1_3_spec.to_markdown(index=False))

In [ ]:
# Flatten results
summary_data = []
for res_list in all_results:
    for res in res_list:
        summary_data.append({
            "Group": res["group"],
            "Method": res["method"],
            "Multiplier": res["sample_multiplier"],

            "Train F1": round(res["train_f1"], 3),
            "CV F1": round(res["cv_f1"], 3),
            "Test F1": round(res["test_f1"], 3),
            "F1 CI Lower": round(res["ci_f1"][0], 3),
            "F1 CI Upper": round(res["ci_f1"][1], 3),
        })

# Convert to DataFrame and display
summary_df_group_1_3_f1 = pd.DataFrame(summary_data)
print(summary_df_group_1_3_f1.to_markdown(index=False))

In [ ]:
# Create the summary list
summary_data = []
for res_list in all_results:
    for res in res_list:
        summary_data.append({
    "Group": res["group"],
    "Method": res["method"],
    "Multiplier": res["sample_multiplier"],
    "Best Params": str(res["best_params"])
        
    })
# Convert to DataFrame
summary_df_group_1_3_bestparams = pd.DataFrame(summary_data)
print(summary_df_group_1_3_bestparams.to_markdown(index=False))   

In [ ]:
from ctgan.synthesizers.ctgan import CTGAN

In [ ]:
# TVAE + RF and TVAE + XGBoost for group_2

def compute_confidence_interval(metric_fn, y_true, y_score, threshold=None, n_bootstraps=1000, ci=0.95):
    stats = []
    y_true = np.array(y_true)
    y_score = np.array(y_score)
    rng = np.random.RandomState(100)

    for _ in range(n_bootstraps):
        indices = rng.randint(0, len(y_true), len(y_true))
        if len(np.unique(y_true[indices])) < 2:
            continue

        if threshold is not None:
            y_pred = (y_score[indices] >= threshold).astype(int)
            value = metric_fn(y_true[indices], y_pred)
        else:
            value = metric_fn(y_true[indices], y_score[indices])

        stats.append(value)

    lower = np.percentile(stats, (1 - ci) / 2 * 100)
    upper = np.percentile(stats, (1 + ci) / 2 * 100)
    return lower, upper

def generate_synthetic_and_run_model(group_columns, df, group_name,
                                     target_col="convert_Within_3Years",
                                     method="RF",  # "RF" or "XGBoost"
                                     test_size=0.3,
                                     random_state=100,
                                     sample_multiplier=[1, 2, 3]):
    # Only allow group_2
    if group_name not in ["group_2"]:
        raise ValueError(f"TVAE augmentation is only supported for 'group_2'. Got: {group_name}")

    real_data_group = pd.concat([df[group_columns], df[target_col]], axis=1)

    train_data, test_data = train_test_split(
        real_data_group,
        test_size=test_size,
        random_state=random_state,
        stratify=real_data_group[target_col]
    )
    
    
    np.random.seed(42)
    torch.manual_seed(42)

    
    metadata = Metadata.detect_from_dataframe(train_data)
    
    

    print(f"Using TVAE for {group_name}")
    tvae_params = {
        'embedding_dim': 64,
        'compress_dims': (256, 128, 64),
        'decompress_dims': (64, 128),
        'l2scale': 1e-5,
        'batch_size': 64,
        'epochs': 100,
        'loss_factor': 0.5,
        'enforce_min_max_values': False,
        'enforce_rounding': False,
        'cuda': True
    }

    synthesizer = TVAESynthesizer(metadata=metadata, **tvae_params)
    synthesizer.fit(train_data)

    results = []

    for mult in sample_multiplier:
        cases_without_dementia = Condition(
            num_rows=int(len(train_data) * mult * 0.5),
            column_values={target_col: 0}
        )
        cases_with_dementia = Condition(
            num_rows=int(len(train_data) * mult * 0.5),
            column_values={target_col: 1})
        synthetic_data = synthesizer.sample_from_conditions(
            conditions=[cases_without_dementia, cases_with_dementia]
        )
        
        # Save synthetic data
        save_dir = r"D:\Singapore\Alzheimer\R files\final data_new\synthetic_data\TVAE\group_2"
        os.makedirs(save_dir, exist_ok=True)
        filename = f"synthetic_{group_name}_TVAE_{mult}x.csv"  # or include method if needed
        save_path = os.path.join(save_dir, filename)
        synthetic_data.to_csv(save_path, index=False)

        augmented_train = pd.concat([train_data, synthetic_data], ignore_index=True)



        X_train_aug = augmented_train[group_columns]
        y_train_aug = augmented_train[target_col]
        X_test = test_data[group_columns]
        y_test = test_data[target_col]

        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=100)

        if method.upper() == "RF":
            model = RandomForestClassifier(random_state=100, class_weight='balanced')
            param_dist = {
                'n_estimators': [150, 200],
                'max_depth': [4, 5],
                'min_samples_split': [10, 15, 20],
                'min_samples_leaf': [6, 8, 10],
                'max_features': ['log2'],
                'bootstrap': [True],
                'max_samples': [0.5, 0.7, 0.9]
            }
        elif method.upper() == "XGBOOST":
            neg_count = (y_train_aug == 0).sum()
            pos_count = (y_train_aug == 1).sum()
            scale_pos_weight = neg_count / pos_count
            model = xgb.XGBClassifier(eval_metric='logloss', random_state=100, scale_pos_weight=scale_pos_weight)
            param_dist = {
                'n_estimators': [50],
                'max_depth': [2, 3, 4],
                'learning_rate': [0.01, 0.02],
                'subsample': [0.6, 0.7, 0.8],
                'colsample_bytree': [0.5, 0.7, 0.9, 1.0],
                'gamma': [0, 0.25, 0.5],
                'reg_alpha': [0, 0.1, 1.0],
                'reg_lambda': [1.0, 1.5, 2.0]
            }
        else:
            raise ValueError(f"Invalid method '{method}'. Choose 'RF' or 'XGBoost'.")

        def specificity_score(y_true, y_pred):
            tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
            return tn / (tn + fp) if (tn + fp) > 0 else 0

        scoring = {
            'roc_auc': 'roc_auc',
            'f1': make_scorer(f1_score),
            'recall': make_scorer(recall_score),
            'specificity': make_scorer(specificity_score)
        }
        search = RandomizedSearchCV(
            model,
            param_distributions=param_dist,
            cv=cv,
            scoring=scoring,
            refit='roc_auc',
            random_state=100
        )

        search.fit(X_train_aug, y_train_aug)
        best_params = search.best_params_

        # Calibration step
        uncalibrated_model = xgb.XGBClassifier(**best_params, random_state=100, scale_pos_weight=scale_pos_weight) if method.upper() == "XGBOOST" else RandomForestClassifier(**best_params, random_state=100, class_weight='balanced')
        calibrated_model = CalibratedClassifierCV(uncalibrated_model, method='isotonic', cv=3)
        calibrated_model.fit(X_train_aug, y_train_aug)

        y_train_proba = calibrated_model.predict_proba(X_train_aug)[:, 1]
        y_test_proba = calibrated_model.predict_proba(X_test)[:, 1]

        # --- Youden threshold ---
        youden_thresholds = []
        for train_idx, val_idx in cv.split(X_train_aug, y_train_aug):
          X_tr, X_val = X_train_aug.iloc[train_idx], X_train_aug.iloc[val_idx]
          y_tr, y_val = y_train_aug.iloc[train_idx], y_train_aug.iloc[val_idx]

          if method.upper() == "XGBOOST":
            model_cv = xgb.XGBClassifier(**best_params, random_state=100, scale_pos_weight=scale_pos_weight)
          else:
            model_cv = RandomForestClassifier(**best_params, random_state=100, class_weight='balanced')
          model_cv_calibrated = CalibratedClassifierCV(model_cv, method='isotonic', cv=3)
          model_cv_calibrated.fit(X_tr, y_tr)

          y_val_proba = model_cv_calibrated.predict_proba(X_val)[:, 1]
          fpr, tpr, thresholds = roc_curve(y_val, y_val_proba)
          youden = tpr - fpr
          best_thresh = thresholds[np.argmax(youden)]
          youden_thresholds.append(best_thresh)

        optimal_threshold = np.mean(youden_thresholds)

        y_train_pred = (y_train_proba >= optimal_threshold).astype(int)
        y_test_pred = (y_test_proba >= optimal_threshold).astype(int)

        # Evaluation metrics
        test_f1 = f1_score(y_test, y_test_pred)
        test_recall = recall_score(y_test, y_test_pred)
        tn_test, fp_test, fn_test, tp_test = confusion_matrix(y_test, y_test_pred).ravel()
        test_specificity = tn_test / (tn_test + fp_test)
        test_auc = roc_auc_score(y_test, y_test_proba)

        train_f1 = f1_score(y_train_aug, y_train_pred)
        train_recall = recall_score(y_train_aug, y_train_pred)
        tn_train, fp_train, fn_train, tp_train = confusion_matrix(y_train_aug, y_train_pred).ravel()
        train_specificity = tn_train / (tn_train + fp_train)
        train_auc = roc_auc_score(y_train_aug, y_train_proba)

        cv_results = search.cv_results_
        best_idx = search.best_index_
        cv_auc = cv_results['mean_test_roc_auc'][best_idx]
        cv_f1 = cv_results['mean_test_f1'][best_idx]
        cv_recall = cv_results['mean_test_recall'][best_idx]
        cv_specificity = cv_results['mean_test_specificity'][best_idx]

        ci_auc = compute_confidence_interval(roc_auc_score, y_test, y_test_proba, threshold=None)
        ci_f1 = compute_confidence_interval(f1_score, y_test, y_test_proba, threshold=optimal_threshold)
        ci_recall = compute_confidence_interval(recall_score, y_test, y_test_proba, threshold=optimal_threshold)
        ci_spec = compute_confidence_interval(specificity_score, y_test, y_test_proba, threshold=optimal_threshold)

        n_bins = 10
        n_bootstrap = 1000
        ci = 99
        bin_edges = np.linspace(0, 1, n_bins + 1)
        bin_centers = (bin_edges[1:] + bin_edges[:-1]) / 2
        bootstrap_curves = []
        rng = np.random.RandomState(100)
        
        for _ in range(n_bootstrap):
            indices = rng.randint(0, len(y_test), len(y_test))
            y_bs = y_test.iloc[indices]
            p_bs = y_test_proba[indices]
            if len(np.unique(y_bs)) < 2:
                continue
            for _ in range(n_bootstrap):
                y_bs, p_bs = resample(y_test, y_test_proba, random_state=None)
            try:
                pt_bs, pp_bs = calibration_curve(y_bs, p_bs, n_bins=n_bins, strategy='uniform')
                interp = np.interp(bin_centers, pp_bs, pt_bs)
                bootstrap_curves.append(interp)
            except:
                continue

        bootstrap_array = np.array(bootstrap_curves)
        lower_bound = np.percentile(bootstrap_array, (100 - ci) / 2, axis=0)
        upper_bound = np.percentile(bootstrap_array, 100 - (100 - ci) / 2, axis=0)

        prob_true, prob_pred = calibration_curve(y_test, y_test_proba, n_bins=n_bins, strategy='uniform')
        plt.figure(figsize=(6, 6))
        plt.plot(prob_pred, prob_true, marker='o', label='Calibrated Model')
        plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Perfect Calibration')
        plt.fill_between(bin_centers, lower_bound, upper_bound, color='blue', alpha=0.2, label='99% CI')
        plt.xlabel('Predicted Probability')
        plt.ylabel('True Probability')
        plt.title(f'Calibration Plot (Test Set) for {group_name}, {method}, Multiplier {mult}')
        plt.legend()
        plt.grid()
        plt.tight_layout()
        plt.savefig(os.path.join(fig_save_dir, f"calibration_{group_name}_{method}_Multiplier_{mult}.pdf"), dpi=600,bbox_inches="tight")
        plt.show()
        plt.close()


        results.append({
            'group': group_name,
            'method': method,
            'sample_multiplier': mult,
            'train_auc': train_auc,
            'cv_auc': cv_auc,
            'test_auc': test_auc,
            'train_f1': train_f1,
            'cv_f1': cv_f1,
            'test_f1': test_f1,
            'train_recall': train_recall,
            'cv_recall': cv_recall,
            'test_recall': test_recall,
            'train_specificity': train_specificity,
            'cv_specificity': cv_specificity,
            'test_specificity': test_specificity,
            'ci_auc': ci_auc,
            'ci_f1': ci_f1,
            'ci_recall': ci_recall,
            'ci_specificity': ci_spec,
            'best_params': search.best_params_
        })

    return results


In [ ]:
results_rf_group_2 = generate_synthetic_and_run_model(
    group_columns=group_2,
    df=df_Data,
    group_name="group_2",
    method="RF"
)

results_xgb_group_2 = generate_synthetic_and_run_model(
    group_columns=group_2,
    df=df_Data,
    group_name="group_2",
    method="XGBoost"
)

In [ ]:
all_results_group_2 = [results_rf_group_2, results_xgb_group_2]

In [ ]:
# Flatten results
summary_data = []
for res_list in all_results_group_2:
    for res in res_list:
        summary_data.append({
            "Group": res["group"],
            "Method": res["method"],
            "Multiplier": res["sample_multiplier"],

            "Train AUC": round(res["train_auc"], 3),
            "CV AUC": round(res["cv_auc"], 3),
            "Test AUC": round(res["test_auc"], 3),
            "AUC CI Lower": round(res["ci_auc"][0], 3),
            "AUC CI Upper": round(res["ci_auc"][1], 3)
        })

# Convert to DataFrame and display
summary_df_group_2_auc = pd.DataFrame(summary_data)
print(summary_df_group_2_auc.to_markdown(index=False))


In [ ]:
# Flatten results
summary_data = []
for res_list in all_results:
    for res in res_list:
        summary_data.append({
            "Group": res["group"],
            "Method": res["method"],
            "Multiplier": res["sample_multiplier"],


            "Train Recall": round(res["train_recall"], 3),
            "CV Recall": round(res["cv_recall"], 3),
            "Test Recall": round(res["test_recall"], 3),
            "Recall CI Lower": round(res["ci_recall"][0], 3),
            "Recall CI Upper": round(res["ci_recall"][1], 3)
        })

# Convert to DataFrame and display
summary_df_group_2_recall = pd.DataFrame(summary_data)
print(summary_df_group_2_recall.to_markdown(index=False))

In [ ]:
# Flatten results
summary_data = []
for res_list in all_results:
    for res in res_list:
        summary_data.append({
            "Group": res["group"],
            "Method": res["method"],
            "Multiplier": res["sample_multiplier"],

            "Train Specificity": round(res["train_specificity"], 3),
            "CV Specificity": round(res["cv_specificity"], 3),
            "Test Specificity": round(res["test_specificity"], 3),
            "Specificity CI Lower": round(res["ci_specificity"][0], 3),
            "Specificity CI Upper": round(res["ci_specificity"][1], 3)
        })

# Convert to DataFrame and display
summary_df_group_2_spec = pd.DataFrame(summary_data)
print(summary_df_group_2_spec.to_markdown(index=False))

In [ ]:
# Flatten results
summary_data = []
for res_list in all_results:
    for res in res_list:
        summary_data.append({
            "Group": res["group"],
            "Method": res["method"],
            "Multiplier": res["sample_multiplier"],

            "Train F1": round(res["train_f1"], 3),
            "CV F1": round(res["cv_f1"], 3),
            "Test F1": round(res["test_f1"], 3),
            "F1 CI Lower": round(res["ci_f1"][0], 3),
            "F1 CI Upper": round(res["ci_f1"][1], 3),
        })

# Convert to DataFrame and display
summary_df_group_2_f1 = pd.DataFrame(summary_data)
print(summary_df_group_2_f1.to_markdown(index=False))

In [ ]:
# Flatten results
summary_data = []
for res_list in all_results:
    for res in res_list:
        summary_data.append({
            "Group": res["group"],
            "Method": res["method"],
            "Multiplier": res["sample_multiplier"],
            "Best Params": str(res["best_params"])
        })

# Convert to DataFrame and display
summary_df_group_2_bestparams = pd.DataFrame(summary_data)
print(summary_df_group_2_bestparams.to_markdown(index=False))

In [ ]:
# TVAE + RF and TVAE + XGBoost for group_4
def compute_confidence_interval(metric_fn, y_true, y_score, threshold=None, n_bootstraps=1000, ci=0.95):
    stats = []
    y_true = np.array(y_true)
    y_score = np.array(y_score)
    rng = np.random.RandomState(100)

    for _ in range(n_bootstraps):
        indices = rng.randint(0, len(y_true), len(y_true))
        if len(np.unique(y_true[indices])) < 2:
            continue

        if threshold is not None:
            y_pred = (y_score[indices] >= threshold).astype(int)
            value = metric_fn(y_true[indices], y_pred)
        else:
            value = metric_fn(y_true[indices], y_score[indices])

        stats.append(value)

    lower = np.percentile(stats, (1 - ci) / 2 * 100)
    upper = np.percentile(stats, (1 + ci) / 2 * 100)
    return lower, upper

def generate_synthetic_and_run_model(group_columns, df, group_name,
                                     target_col="convert_Within_3Years",
                                     method="RF",  # "RF" or "XGBoost"
                                     test_size=0.3,
                                     random_state=100,
                                     sample_multiplier=[1, 2, 3]):
    # Only allow group_4
    if group_name not in ["group_4"]:
        raise ValueError(f"TVAE augmentation is only supported for 'group_4'. Got: {group_name}")

    # Combine predictors and target
    real_data_group = pd.concat([df[group_columns], df[target_col]], axis=1)

    # Train/test split
    train_data, test_data = train_test_split(
        real_data_group,
        test_size=test_size,
        random_state=random_state,
        stratify=real_data_group[target_col]
    )

    # Reproducibility
    np.random.seed(42)
    torch.manual_seed(42)

    # TVAE metadata + config
    metadata = Metadata.detect_from_dataframe(train_data)

    print(f"Using TVAE for {group_name}")
    tvae_params = {
        'embedding_dim': 128,
        'compress_dims': (256, 128),
        'decompress_dims': (64, 128),
        'l2scale': 1e-4,
        'batch_size': 64,
        'epochs': 150,
        'loss_factor': 2,
        'enforce_min_max_values': True,
        'enforce_rounding': True,
        'cuda': True
    }

    synthesizer = TVAESynthesizer(metadata=metadata, **tvae_params)
    synthesizer.fit(train_data)

    results = []

    for mult in sample_multiplier:
        synthetic_data = synthesizer.sample(num_rows=len(train_data) * mult)
        # Save synthetic data
        save_dir = r"D:\Singapore\Alzheimer\R files\final data_new\synthetic_data\TVAE\group_4"
        os.makedirs(save_dir, exist_ok=True)
        filename = f"synthetic_{group_name}_TVAE_{mult}x.csv"  # or include method if needed
        save_path = os.path.join(save_dir, filename)
        synthetic_data.to_csv(save_path, index=False)

        augmented_train = pd.concat([train_data, synthetic_data], ignore_index=True)

        X_train_aug = augmented_train[group_columns]
        y_train_aug = augmented_train[target_col]
        X_test = test_data[group_columns]
        y_test = test_data[target_col]

        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=100)

        if method.upper() == "RF":
            model = RandomForestClassifier(random_state=100, class_weight='balanced')
            param_dist = {
                'n_estimators': [150, 200, 300],
                'max_depth': [4, 5],
                'min_samples_split': [10, 15, 20],
                'min_samples_leaf': [6, 8, 10],
                'max_features': ['sqrt'],
                'bootstrap': [True],
                'max_samples': [0.7, 0.8, 0.9]
            }
        elif method.upper() == "XGBOOST":
            neg_count = (y_train_aug == 0).sum()
            pos_count = (y_train_aug == 1).sum()
            scale_pos_weight = neg_count / pos_count
            model = xgb.XGBClassifier(eval_metric='logloss', random_state=100, scale_pos_weight=scale_pos_weight)
            param_dist = {
                'n_estimators': [50],
                'max_depth': [2, 3],
                'learning_rate': [0.005, 0.01],
                'subsample': [0.6, 0.7, 0.8],
                'colsample_bytree': [0.5, 0.7, 0.9, 1.0],
                'gamma': [0.1],
                'reg_alpha': [1, 5],
                'reg_lambda': [1.0, 1.5, 2.0]
            }
        else:
            raise ValueError(f"Invalid method '{method}'. Choose 'RF' or 'XGBoost'.")


        def specificity_score(y_true, y_pred):
            tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
            return tn / (tn + fp) if (tn + fp) > 0 else 0

        scoring = {
            'roc_auc': 'roc_auc',
            'f1': make_scorer(f1_score),
            'recall': make_scorer(recall_score),
            'specificity': make_scorer(specificity_score)
        }
        search = RandomizedSearchCV(
            model,
            param_distributions=param_dist,
            cv=cv,
            scoring=scoring,  # Use the full dict here
            refit='roc_auc',  # Refit using ROC AUC
            random_state=100
        )


        search.fit(X_train_aug, y_train_aug)
        best_params = search.best_params_

        # Calibration step
        uncalibrated_model = xgb.XGBClassifier(**best_params, random_state=100, scale_pos_weight=scale_pos_weight) if method.upper() == "XGBOOST" else RandomForestClassifier(**best_params, random_state=100, class_weight='balanced')
        calibrated_model = CalibratedClassifierCV(uncalibrated_model,  method='isotonic', cv=3)
        calibrated_model.fit(X_train_aug, y_train_aug)

        y_train_proba = calibrated_model.predict_proba(X_train_aug)[:, 1]
        y_test_proba = calibrated_model.predict_proba(X_test)[:, 1]

        # --- Youden threshold ---
        youden_thresholds = []
        for train_idx, val_idx in cv.split(X_train_aug, y_train_aug):
          X_tr, X_val = X_train_aug.iloc[train_idx], X_train_aug.iloc[val_idx]
          y_tr, y_val = y_train_aug.iloc[train_idx], y_train_aug.iloc[val_idx]

          if method.upper() == "XGBOOST":
            model_cv = xgb.XGBClassifier(**best_params, random_state=100, scale_pos_weight=scale_pos_weight)
          else:
            model_cv = RandomForestClassifier(**best_params, random_state=100, class_weight='balanced')
          model_cv_calibrated = CalibratedClassifierCV(model_cv,  method='isotonic', cv=3)
          model_cv_calibrated.fit(X_tr, y_tr)

          y_val_proba = model_cv_calibrated.predict_proba(X_val)[:, 1]
          fpr, tpr, thresholds = roc_curve(y_val, y_val_proba)
          youden = tpr - fpr
          best_thresh = thresholds[np.argmax(youden)]
          youden_thresholds.append(best_thresh)

        optimal_threshold = np.mean(youden_thresholds)

        y_train_pred = (y_train_proba >= optimal_threshold).astype(int)
        y_test_pred = (y_test_proba >= optimal_threshold).astype(int)

        # Evaluation metrics

        test_f1 = f1_score(y_test, y_test_pred)
        test_recall = recall_score(y_test, y_test_pred)
        tn_test, fp_test, fn_test, tp_test = confusion_matrix(y_test, y_test_pred).ravel()
        test_specificity = tn_test / (tn_test + fp_test)
        test_auc = roc_auc_score(y_test, y_test_proba)

        train_f1 = f1_score(y_train_aug, y_train_pred)
        train_recall = recall_score(y_train_aug, y_train_pred)
        tn_train, fp_train, fn_train, tp_train = confusion_matrix(y_train_aug, y_train_pred).ravel()
        train_specificity = tn_train / (tn_train + fp_train)
        train_auc = roc_auc_score(y_train_aug, y_train_proba)

        cv_results = search.cv_results_
        best_idx = search.best_index_
        cv_auc = cv_results['mean_test_roc_auc'][best_idx]
        cv_f1 = cv_results['mean_test_f1'][best_idx]
        cv_recall = cv_results['mean_test_recall'][best_idx]
        cv_specificity = cv_results['mean_test_specificity'][best_idx]

        ci_auc = compute_confidence_interval(roc_auc_score, y_test, y_test_proba, threshold=None)
        ci_f1 = compute_confidence_interval(f1_score, y_test, y_test_proba, threshold=optimal_threshold)
        ci_recall = compute_confidence_interval(recall_score, y_test, y_test_proba, threshold=optimal_threshold)
        ci_spec = compute_confidence_interval(specificity_score, y_test, y_test_proba, threshold=optimal_threshold)

        n_bins = 10
        n_bootstrap = 1000
        ci = 99
        bin_edges = np.linspace(0, 1, n_bins + 1)
        bin_centers = (bin_edges[1:] + bin_edges[:-1]) / 2
        bootstrap_curves = []

        for _ in range(n_bootstrap):
            y_bs, p_bs = resample(y_test, y_test_proba, random_state=None)
            try:
                pt_bs, pp_bs = calibration_curve(y_bs, p_bs, n_bins=n_bins, strategy='uniform')
                interp = np.interp(bin_centers, pp_bs, pt_bs)
                bootstrap_curves.append(interp)
            except:
                continue

        bootstrap_array = np.array(bootstrap_curves)
        lower_bound = np.percentile(bootstrap_array, (100 - ci) / 2, axis=0)
        upper_bound = np.percentile(bootstrap_array, 100 - (100 - ci) / 2, axis=0)

        prob_true, prob_pred = calibration_curve(y_test, y_test_proba, n_bins=n_bins, strategy='uniform')
        plt.figure(figsize=(6, 6))
        plt.plot(prob_pred, prob_true, marker='o', label='Calibrated Model')
        plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Perfect Calibration')
        plt.fill_between(bin_centers, lower_bound, upper_bound, color='blue', alpha=0.2, label='99% CI')
        plt.xlabel('Predicted Probability')
        plt.ylabel('True Probability')
        plt.title(f'Calibration Plot (Test Set) for {group_name}, {method}, Multiplier {mult}')
        plt.legend()
        plt.grid()
        plt.tight_layout()
        plt.savefig(f"{fig_save_dir}/calibration_{group_name}_{method}_Multiplier {mult}.pdf",dpi=600,bbox_inches="tight")
        plt.show()
        plt.close()


        results.append({
            'group': group_name,
            'method': method,
            'sample_multiplier': mult,
            'train_auc': train_auc,
            'cv_auc': cv_auc,
            'test_auc': test_auc,
            'train_f1': train_f1,
            'cv_f1': cv_f1,
            'test_f1': test_f1,
            'train_recall': train_recall,
            'cv_recall': cv_recall,
            'test_recall': test_recall,
            'train_specificity': train_specificity,
            'cv_specificity': cv_specificity,
            'test_specificity': test_specificity,
            'ci_auc': ci_auc,
            'ci_f1': ci_f1,
            'ci_recall': ci_recall,
            'ci_specificity': ci_spec,
            'best_params': search.best_params_
        })

    return results


In [ ]:
results_rf_group_4 = generate_synthetic_and_run_model(
    group_columns=group_4,
    df=df_Data,
    group_name="group_4",
    method="RF"
)

results_xgb_group_4 = generate_synthetic_and_run_model(
    group_columns=group_4,
    df=df_Data,
    group_name="group_4",
    method="XGBoost"
)

In [ ]:
all_results = [results_rf_group_4, results_xgb_group_4]

In [ ]:
# Flatten results
summary_data = []
for res_list in all_results:
    for res in res_list:
        summary_data.append({
            "Group": res["group"],
            "Method": res["method"],
            "Multiplier": res["sample_multiplier"],

            "Train AUC": round(res["train_auc"], 3),
            "CV AUC": round(res["cv_auc"], 3),
            "Test AUC": round(res["test_auc"], 3),
            "AUC CI Lower": round(res["ci_auc"][0], 3),
            "AUC CI Upper": round(res["ci_auc"][1], 3)

        })

# Convert to DataFrame and display
summary_df_group_4_auc = pd.DataFrame(summary_data)
print(summary_df_group_4_auc.to_markdown(index=False))

In [ ]:
# Flatten results
summary_data = []
for res_list in all_results:
    for res in res_list:
        summary_data.append({
            "Group": res["group"],
            "Method": res["method"],
            "Multiplier": res["sample_multiplier"],

            "Train Recall": round(res["train_recall"], 3),
            "CV Recall": round(res["cv_recall"], 3),
            "Test Recall": round(res["test_recall"], 3),
            "Recall CI Lower": round(res["ci_recall"][0], 3),
            "Recall CI Upper": round(res["ci_recall"][1], 3)
        })

# Convert to DataFrame and display
summary_df_group_4_recall = pd.DataFrame(summary_data)
print(summary_df_group_4_recall.to_markdown(index=False))

In [ ]:
# Flatten results
summary_data = []
for res_list in all_results:
    for res in res_list:
        summary_data.append({
            "Group": res["group"],
            "Method": res["method"],
            "Multiplier": res["sample_multiplier"],

            "Train Specificity": round(res["train_specificity"], 3),
            "CV Specificity": round(res["cv_specificity"], 3),
            "Test Specificity": round(res["test_specificity"], 3),
            "Specificity CI Lower": round(res["ci_specificity"][0], 3),
            "Specificity CI Upper": round(res["ci_specificity"][1], 3)

        })

# Convert to DataFrame and display
summary_df_group_4_spec = pd.DataFrame(summary_data)
print(summary_df_group_4_spec.to_markdown(index=False))

In [ ]:
# Flatten results
summary_data = []
for res_list in all_results:
    for res in res_list:
        summary_data.append({
            "Group": res["group"],
            "Method": res["method"],
            "Multiplier": res["sample_multiplier"],


            "Train F1": round(res["train_f1"], 3),
            "CV F1": round(res["cv_f1"], 3),
            "Test F1": round(res["test_f1"], 3),
            "F1 CI Lower": round(res["ci_f1"][0], 3),
            "F1 CI Upper": round(res["ci_f1"][1], 3)

        })

# Convert to DataFrame and display
summary_df_group_4_f1 = pd.DataFrame(summary_data)
print(summary_df_group_4_f1.to_markdown(index=False))

In [ ]:
# Flatten results
summary_data = []
for res_list in all_results:
    for res in res_list:
        summary_data.append({
            "Group": res["group"],
            "Method": res["method"],
            "Multiplier": res["sample_multiplier"],
            "Best Params": str(res["best_params"])
        })

# Convert to DataFrame and display
summary_df_group_4_bestparams = pd.DataFrame(summary_data)
print(summary_df_group_4_bestparams.to_markdown(index=False))

In [ ]:

import seaborn as sns


sns.set_style("whitegrid")
sns.set_context("talk", font_scale=1.2)

# =====================================================
# Helper function → unify column names
# =====================================================
def prepare_metric(df, metric,
                   value_col,
                   lower_col,
                   upper_col):

    return df.rename(columns={
        value_col: "Value",
        lower_col: "Lower",
        upper_col: "Upper"
    }).assign(Metric=metric)


# =====================================================
# Build summary_all (STANDARDIZED)
# =====================================================
summary_all = pd.concat([

    prepare_metric(summary_df_group_1_3_auc,
                   "AUC",
                   "Test AUC",
                   "AUC CI Lower",
                   "AUC CI Upper"),

    prepare_metric(summary_df_group_1_3_recall,
                   "Recall",
                   "Test Recall",
                   "Recall CI Lower",
                   "Recall CI Upper"),

    prepare_metric(summary_df_group_1_3_spec,
                   "Specificity",
                   "Test Specificity",
                   "Specificity CI Lower",
                   "Specificity CI Upper"),

    prepare_metric(summary_df_group_1_3_f1,
                   "F1",
                   "Test F1",
                   "F1 CI Lower",
                   "F1 CI Upper"),

    prepare_metric(summary_df_group_2_auc,
                   "AUC",
                   "Test AUC",
                   "AUC CI Lower",
                   "AUC CI Upper"),

    prepare_metric(summary_df_group_2_recall,
                   "Recall",
                   "Test Recall",
                   "Recall CI Lower",
                   "Recall CI Upper"),

    prepare_metric(summary_df_group_2_spec,
                   "Specificity",
                   "Test Specificity",
                   "Specificity CI Lower",
                   "Specificity CI Upper"),

    prepare_metric(summary_df_group_2_f1,
                   "F1",
                   "Test F1",
                   "F1 CI Lower",
                   "F1 CI Upper"),

    prepare_metric(summary_df_group_4_auc,
                   "AUC",
                   "Test AUC",
                   "AUC CI Lower",
                   "AUC CI Upper"),

    prepare_metric(summary_df_group_4_recall,
                   "Recall",
                   "Test Recall",
                   "Recall CI Lower",
                   "Recall CI Upper"),

    prepare_metric(summary_df_group_4_spec,
                   "Specificity",
                   "Test Specificity",
                   "Specificity CI Lower",
                   "Specificity CI Upper"),

    prepare_metric(summary_df_group_4_f1,
                   "F1",
                   "Test F1",
                   "F1 CI Lower",
                   "F1 CI Upper"),

], ignore_index=True)


# =====================================================
# Labels
# =====================================================
summary_all["Label"] = (
    summary_all["Group"].str.replace("group_", "G", regex=False)
    + "-"
    + summary_all["Method"]
    + "-M"
    + summary_all["Multiplier"].astype(str)
)

metrics = ["AUC", "Recall", "Specificity", "F1"]


# =====================================================
# Plot
# =====================================================
fig, axes = plt.subplots(
    2, 2,
    figsize=(20, 20),
    constrained_layout=True
)

axes = axes.flatten()

for i, metric in enumerate(metrics):

    ax = axes[i]

    data = (
        summary_all[summary_all["Metric"] == metric]
        .copy()
        .sort_values("Label")
    )

    sns.scatterplot(
        data=data,
        x="Value",
        y="Label",
        ax=ax,
        s=60,
        color="tab:blue"
    )

    # Confidence intervals
    for _, row in data.iterrows():
        ax.plot(
            [row["Lower"], row["Upper"]],
            [row["Label"], row["Label"]],
            color="black",
            linewidth=1.5
        )

    ax.set_title(metric)
    ax.set_ylabel("")
    ax.grid(True, axis="x", alpha=0.4)

fig.subplots_adjust(left=0.30)
plt.savefig(f"{fig_save_dir}/CI.pdf",dpi=600,bbox_inches="tight")
plt.show()

In [ ]:
    
    for mult in sample_multiplier:
        n_total = len(train_data) * mult
        synthetic_data = synthesizer.sample(num_rows=n_total)
        class1_synth = synthetic_data[synthetic_data[target_col] == 1]
        if len(class1_synth) == 0:
            print(f"No class 1 in synthetic data for mult={mult}, injecting real class 1 samples...")
            minority_ratio = len(train_data[train_data[target_col] == 1]) / len(train_data)
            num_class1 = int(n_total * minority_ratio)
            class1_injected = train_data[train_data[target_col] == 1].sample(n=num_class1,
                                                                             replace=True,
                                                                             random_state=random_state)
    
            synthetic_data = pd.concat([synthetic_data, class1_injected], ignore_index=True)
        # Shuffle the synthetic data (optional but can ensure randomness)
        synthetic_data = synthetic_data.sample(frac=1, random_state=random_state).reset_index(drop=True)

        # Ensure we still have the correct number of synthetic data (just in case)
        synthetic_data = synthetic_data.head(n_total)     

            
        save_dir = r"D:\Singapore\Alzheimer\R files\final data_new\synthetic_data\group_2"
        os.makedirs(save_dir, exist_ok=True)
        filename = f"synthetic_{group_name}_TVAE_{mult}x.csv"
        save_path = os.path.join(save_dir, filename)
        synthetic_data.to_csv(save_path, index=False)

        augmented_train = pd.concat([train_data, synthetic_data], ignore_index=True)
'''